In [17]:
import pandas as pd
import numpy as np
from apyori import apriori
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

In [39]:
df = pd.read_csv('Market_Basket_Optimisation.csv', header=None)
df.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,shrimp,almonds,avocado,vegetables mix,green grapes,whole weat flour,yams,cottage cheese,energy drink,tomato juice,low fat yogurt,green tea,honey,salad,mineral water,salmon,antioxydant juice,frozen smoothie,spinach,olive oil
1,burgers,meatballs,eggs,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,chutney,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,turkey,avocado,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,mineral water,milk,energy bar,whole wheat rice,green tea,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7501 entries, 0 to 7500
Data columns (total 20 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   0       7501 non-null   object
 1   1       5747 non-null   object
 2   2       4389 non-null   object
 3   3       3345 non-null   object
 4   4       2529 non-null   object
 5   5       1864 non-null   object
 6   6       1369 non-null   object
 7   7       981 non-null    object
 8   8       654 non-null    object
 9   9       395 non-null    object
 10  10      256 non-null    object
 11  11      154 non-null    object
 12  12      87 non-null     object
 13  13      47 non-null     object
 14  14      25 non-null     object
 15  15      8 non-null      object
 16  16      4 non-null      object
 17  17      4 non-null      object
 18  18      3 non-null      object
 19  19      1 non-null      object
dtypes: object(20)
memory usage: 1.1+ MB


In [ ]:
# Kiểm tra dữ liệu thiếu


In [40]:
# Thay thế giá trị rỗng bằng 0.
df.fillna(0,inplace=True)
df.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,shrimp,almonds,avocado,vegetables mix,green grapes,whole weat flour,yams,cottage cheese,energy drink,tomato juice,low fat yogurt,green tea,honey,salad,mineral water,salmon,antioxydant juice,frozen smoothie,spinach,olive oil
1,burgers,meatballs,eggs,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,chutney,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,turkey,avocado,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,mineral water,milk,energy bar,whole wheat rice,green tea,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [48]:
# Chuyển đổi dữ liệu sang dạng list of lists

transactions = []

for i in range(0,len(df)):
    transactions.append([str(df.values[i,j]) 
for j in range(0,20) 
if str(df.values[i,j])!='0'])

print(f"\nTổng transactions: {len(transactions)}")
print(f"Mẫu transactions: {transactions[0:5]}")


Tổng transactions: 7501
Mẫu transactions: [['shrimp', 'almonds', 'avocado', 'vegetables mix', 'green grapes', 'whole weat flour', 'yams', 'cottage cheese', 'energy drink', 'tomato juice', 'low fat yogurt', 'green tea', 'honey', 'salad', 'mineral water', 'salmon', 'antioxydant juice', 'frozen smoothie', 'spinach', 'olive oil'], ['burgers', 'meatballs', 'eggs'], ['chutney'], ['turkey', 'avocado'], ['mineral water', 'milk', 'energy bar', 'whole wheat rice', 'green tea']]


In [23]:
transactions[0]

['shrimp',
 'almonds',
 'avocado',
 'vegetables mix',
 'green grapes',
 'whole weat flour',
 'yams',
 'cottage cheese',
 'energy drink',
 'tomato juice',
 'low fat yogurt',
 'green tea',
 'honey',
 'salad',
 'mineral water',
 'salmon',
 'antioxydant juice',
 'frozen smoothie',
 'spinach',
 'olive oil']

In [24]:
transactions[1]

['burgers', 'meatballs', 'eggs']

In [25]:
transactions[2]

['chutney']

In [43]:
# Thống kê số items trong mỗi transaction
transaction_lengths = [len(t) for t in transactions]
print(f"\nThống kê độ dài Transaction:")
print(f"Min items: {min(transaction_lengths)}")
print(f"Max items: {max(transaction_lengths)}")
print(f"Average items: {np.mean(transaction_lengths):.2f}")


Thống kê độ dài Transaction:
Min items: 1
Max items: 20
Average items: 3.91


In [ ]:
rules = apriori(
    transactions, 
    min_support=0.003, 
    min_confidence=0.2, 
    min_lift=3, 
    min_length=2,
    max_length=2 # Giới hạn ở luật 2 thành phần để dễ phân tích ban đầu
)

# Chuyển đổi kết quả từ generator sang list để xử lý
results = list(rules)
print(f"Số lượng luật kết hợp tìm thấy: {len(results)}")

Số lượng luật kết hợp tìm thấy: 9


In [ ]:
# Chuyển đổi kết quả thành DataFrame
def inspect_apyori_results(results):
    lhs         = []
    rhs         = []
    supports    = []
    confidences = []
    lifts       = []
    
    for result in results:
        support_val = result.support
        ordered_stats = result.ordered_statistics
        
        for stat in ordered_stats:
            # stat: items_base (tiền đề)
            # stat: items_add (hệ quả)
            # stat: confidence
            # stat: lift
            
            # Chỉ lấy các luật có tiền đề và hệ quả không rỗng
            if stat.items_base and stat.items_add:
                lhs.append(list(stat.items_base))
                rhs.append(list(stat.items_add))
                supports.append(support_val)
                confidences.append(stat.confidence)
                lifts.append(stat.lift)
    
    return pd.DataFrame({
        'LHs': lhs,
        'RHs': rhs,
        'Support': supports,
        'Confidence': confidences,
        'Lift': lifts
    })

# Tạo DataFrame kết quả
results_df = inspect_apyori_results(results)


In [47]:
print(results_df.head(10))

                      LHs            RHs   Support  Confidence      Lift
0           [light cream]      [chicken]  0.004533    0.290598  4.843951
1  [mushroom cream sauce]     [escalope]  0.005733    0.300699  3.790833
2                 [pasta]     [escalope]  0.005866    0.372881  4.700812
3         [fromage blanc]        [honey]  0.003333    0.245098  5.164271
4         [herb & pepper]  [ground beef]  0.015998    0.323450  3.291994
5          [tomato sauce]  [ground beef]  0.005333    0.377358  3.840659
6           [light cream]    [olive oil]  0.003200    0.205128  3.114710
7     [whole wheat pasta]    [olive oil]  0.007999    0.271493  4.122410
8                 [pasta]       [shrimp]  0.005066    0.322034  4.506672


In [51]:
# Sắp xếp theo Lift giảm dần để thấy các luật mạnh nhất
top_rules = results_df.nlargest(10, 'Lift')

print("\nTop 10 Luật Kết Hợp Có Độ Nâng (Lift) Cao Nhất:")
print(top_rules.to_string(index=False))


Top 10 Luật Kết Hợp Có Độ Nâng (Lift) Cao Nhất:
                   LHs           RHs  Support  Confidence     Lift
       [fromage blanc]       [honey] 0.003333    0.245098 5.164271
         [light cream]     [chicken] 0.004533    0.290598 4.843951
               [pasta]    [escalope] 0.005866    0.372881 4.700812
               [pasta]      [shrimp] 0.005066    0.322034 4.506672
   [whole wheat pasta]   [olive oil] 0.007999    0.271493 4.122410
        [tomato sauce] [ground beef] 0.005333    0.377358 3.840659
[mushroom cream sauce]    [escalope] 0.005733    0.300699 3.790833
       [herb & pepper] [ground beef] 0.015998    0.323450 3.291994
         [light cream]   [olive oil] 0.003200    0.205128 3.114710
